In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import torchvision

print("Loaded Imports")

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?  At each training iteration, process 64 sequences simultaneously.
block_size = 256 # It is the maximum context length.The model can look at at most 256 previous tokens at a time.
max_iters = 5000   #Train for 5,000 iterations.
#Every 500 iterations, evaluate the model.This helps determine whether the model is improving.
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200   # When evaluating, run 200 batches and average their loss. This gives a less noisy estimate than using one batch
n_embd = 384    # embedding dimension.Every token eventually becomes a vector of length 384.
n_head = 6      # 6 attention heads.Each head can learn different relationships.
n_layer = 6     # six Transformer blocks stacked one after another
dropout = 0.2
# ------------

IMG_SIZE = 16   # downsample CIFAR-100's 32x32 to 16x16
K = 32          # quantized gray levels per pixel
BOS = K         # beginning-of-image token id

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# DATA Loading

vocab_size = K + 1

def prepare_tokens(cifar_dataset, K=K, img_size=IMG_SIZE):
    data = cifar_dataset.data.astype(np.float32)
    gray = 0.299 * data[..., 0] + 0.587 * data[..., 1] + 0.114 * data[..., 2]
    n = gray.shape[0]
    gray = gray.reshape(n, img_size, 2, img_size, 2).mean(axis=(2, 4))
    q = np.round(gray / 255.0 * (K - 1)).astype(np.int64)
    # each pixel's brightness (0–255) is rescaled into K=32 discrete gray levels using round(gray/255 * 31)
    return torch.from_numpy(q.reshape(n, img_size * img_size))

train_set = torchvision.datasets.CIFAR100(root='./data', train=True, download=True)
val_set = torchvision.datasets.CIFAR100(root='./data', train=False, download=True)
train_tokens = prepare_tokens(train_set)
val_tokens = prepare_tokens(val_set)

# data loading
def get_batch(split):
    tokens = train_tokens if split == 'train' else val_tokens
    ix = torch.randint(len(tokens), (batch_size,))
    imgs = tokens[ix]
    bos_col = torch.full((batch_size, 1), BOS, dtype=torch.long)
    #predict the next token" setup as text GPT: given BOS, p1, p2, ..., p254, predict p1, p2, ..., p255
    seq = torch.cat([bos_col, imgs], dim=1)
    x = seq[:, :-1]
    y = seq[:, 1:]
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()   # Average all 200 losses.
    model.train()          # Put the model back into training mode.
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)   # take the input numbers, multiply them by a set of learnable weights, and add a bias.
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)  # Converts each embedding into a key, query, value vector.
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))  # creates a lower-triangular matrix.

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5   # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)  future token gets: -inf
        wei = F.softmax(wei, dim=-1) # (B, T, T)   Convert attention scores into probabilities, model has learned how much attention to give each token
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)  attention weights x value vectors = context-aware representation
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) #6 heads
        self.proj = nn.Linear(head_size * num_heads, n_embd)  # 64 x 6 = 384
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)    # six heads and concatenate their outputs.
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),    # Every negative value in the 1536-dimensional tensor is instantly set to zero.
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),   # randomly zeroes some fraction (dropout = 0.2) of the output values during training only, as regularization.
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation
    MultiHeadAttention handles the communication (tokens looking at each other),
    while the FeedFoward class handles the computation (tokens thinking independently about what they just saw)."""

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)  # LayerNorm normalizes each token's vector (zero mean, unit variance) before it enters attention/feedforward.
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)   # Each character index gets mapped to a learned 384-dim vector.
        # Initially random; the model learns meaningful embeddings during training.

        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)


    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
        # A logit is basically a raw score for every possible next token.

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)   # model gets a score for how well it predicted.

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]  #model can only look at the latest 256 tokens.
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)   new token becomes part of the context.
        return idx

model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)   # clears gradients from the previous iteration
    loss.backward()
    optimizer.step()    # learning i.e prediction error > gradient > change weights >better prediction next time

# generate from the model
def tokens_to_image(tokens, K=K, img_size=IMG_SIZE):
    pixels = (tokens.float() / (K - 1) * 255.0).clamp(0, 255).byte()
    return pixels.view(img_size, img_size).cpu().numpy()

context = torch.full((1, 1), BOS, dtype=torch.long, device=device)
generated = m.generate(context, max_new_tokens=block_size)[0][1:]
img = tokens_to_image(generated)

import matplotlib.pyplot as plt
plt.imsave('generated_cifar_image.png', img, cmap='gray')
print("generated image")

Loaded Imports


100%|██████████| 169M/169M [15:31<00:00, 181kB/s]


10.764321 M parameters
step 0: train loss 3.5019, val loss 3.5027
step 500: train loss 2.0502, val loss 2.0607
step 1000: train loss 1.9927, val loss 2.0027
step 1500: train loss 1.9509, val loss 1.9643
step 2000: train loss 1.9258, val loss 1.9347
step 2500: train loss 1.9085, val loss 1.9233
step 3000: train loss 1.9092, val loss 1.9218
step 3500: train loss 1.8959, val loss 1.9092
step 4000: train loss 1.8921, val loss 1.9083
step 4500: train loss 1.8897, val loss 1.9060
step 4999: train loss 1.8834, val loss 1.8963
generated image
